In [0]:
# Row counts
for t in ["gold.dim_date","gold.dim_location","gold.dim_commodity",
          "gold.fact_food_price","gold.fact_food_price_index"]:
    print(t, spark.table(t).count())

# Fact grain must be unique
dups = (spark.table("gold.fact_food_price")
        .groupBy("location_key","date_key","commodity_key").count().filter("count > 1").count())
print("Duplicate fact keys:", dups)

# Logging evidence
display(spark.sql("SELECT * FROM gold.pipeline_step_metrics ORDER BY start_time DESC"))
display(spark.sql("SELECT * FROM audit.pipeline_runs WHERE layer='Gold' ORDER BY start_time DESC"))

# Sample star-schema query (use as a Spark UI / DAG target)
display(spark.sql("""
  SELECT l.country, d.year_month, c.commodity_group, round(avg(f.price),2) AS avg_price
  FROM gold.fact_food_price f
  JOIN gold.dim_location  l ON f.location_key  = l.location_key
  JOIN gold.dim_date      d ON f.date_key      = d.date_key
  JOIN gold.dim_commodity c ON f.commodity_key = c.commodity_key
  WHERE d.year = 2024
  GROUP BY l.country, d.year_month, c.commodity_group
  ORDER BY l.country, d.year_month"""))

# Proof of clustering
display(spark.sql("DESCRIBE DETAIL gold.fact_food_price"))   # check the clusteringColumns field

# Data dictionary for the README
def dictionary(table, pk, fk=None):
    fk = fk or {}
    print(f"\n### {table}\n\n| Column | Type | Key |\n|---|---|---|")
    for f in spark.table(table).schema.fields:
        key = "PK" if f.name in pk else (f"FK -> {fk[f.name]}" if f.name in fk else "")
        print(f"| {f.name} | {f.dataType.simpleString()} | {key} |")

dictionary("gold.dim_date", ["date_key"])
dictionary("gold.dim_location", ["location_key"])
dictionary("gold.dim_commodity", ["commodity_key"])
dictionary("gold.fact_food_price", ["location_key","date_key","commodity_key"],
           {"location_key":"dim_location","date_key":"dim_date","commodity_key":"dim_commodity"})
dictionary("gold.fact_food_price_index", ["location_key","date_key"],
           {"location_key":"dim_location","date_key":"dim_date"})

In [0]:
spark.sql("SHOW VIEWS IN gold").show(truncate=False)

In [0]:
display(spark.sql("""
    SELECT *
    FROM gold.vw_silver_rejects_by_reason
    ORDER BY rejected_rows DESC
"""))

In [0]:
display(spark.sql("""
    SELECT *
    FROM gold.vw_gold_step_metrics
"""))

In [0]:
from pyspark.sql import functions as F

silver = spark.table("silver.world_bank_food_prices")

(silver
    .agg(
        F.count(
            F.when(
                F.col("rice").isNotNull() &
                F.col("c_rice").isNotNull(), 1
            )
        ).alias("both_available"),

        F.count(
            F.when(
                F.col("rice").isNotNull() &
                F.col("c_rice").isNotNull() &
                (F.col("rice") == F.col("c_rice")), 1
            )
        ).alias("equal_values"),

        F.count(
            F.when(
                F.col("rice").isNotNull() &
                F.col("c_rice").isNotNull() &
                (F.col("rice") != F.col("c_rice")), 1
            )
        ).alias("different_values")
    )
    .show()
)